In [1]:
#import library
import os
import rasterio
from rasterio.mask import mask
import geopandas as gpd
from shapely.geometry import mapping
from tqdm import tqdm

## Clip Formes

In [2]:
# Define the paths for the source raster, training polygons, and output directory
raster_path = r"D:\thesis\13 After Yudis\GitHub\Raster\sentinel_medan.tif"
polygon_path = r"D:\thesis\13 After Yudis\GitHub\Shp\train_polygon_28Nov_DL.shp"
output_root = r"D:\thesis\13 After Yudis\GitHub\clip_hasil"
label_col = "kelas"  # Column name containing the class labels

In [3]:
# Load the shapefile containing the polygons using GeoPandas
gdf = gpd.read_file(polygon_path)
# Open the raster file using rasterio
raster = rasterio.open(raster_path)

In [4]:
# Define the paths for the source raster, training polygons, and output directory
for idx, row in tqdm(gdf.iterrows(), total=len(gdf)):
    # class name (replace spaces with underscores to be safe for folder names)
    kelas = str(row[label_col]).replace(" ", "_")
    geom = [mapping(row["geometry"])]

    # create folder for each class and subfolder for images
    kelas_dir = os.path.join(output_root, kelas)
    image_dir = os.path.join(kelas_dir, "images")
    os.makedirs(image_dir, exist_ok=True)

    try:
        # Clip raster according to polygon
        out_image, out_transform = mask(raster, geom, crop=True)
        out_meta = raster.meta.copy()

        # Clip raster according to polygon
        out_meta.update({
            "driver": "GTiff",
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform
        })

        # Save the clip results to the images folder
        out_name = f"poly_{idx:04d}.tif"
        out_path = os.path.join(image_dir, out_name)
        with rasterio.open(out_path, "w", **out_meta) as dest:
            dest.write(out_image)

    except Exception as e:
        print(f"Polygon {idx} is skipped (error: {e})")

raster.close()
print("All polygons are successfully clipped and saved in the class/images/folder.")

100%|██████████| 8885/8885 [01:20<00:00, 110.30it/s]

All polygons are successfully clipped and saved in the class/images/folder.


## Clip Background


In [6]:
# Define the paths for the source raster, training polygons, and output directory
raster_path = r"D:\thesis\13 After Yudis\GitHub\Raster\sentinel_medan.tif"
background_path = r"D:\thesis\13 After Yudis\GitHub\Shp\background.shp"
output_root = r"D:\thesis\13 After Yudis\GitHub\clip_background"
label_col = "class"  # Column name containing the class labels

In [7]:
# Load the shapefile containing the polygons using GeoPandas
gdf = gpd.read_file(background_path)
# Open the raster file using rasterio
raster = rasterio.open(raster_path)

In [8]:
#Loop fuction for clip each polygon background
for idx, row in tqdm(gdf.iterrows(), total=len(gdf)):
    kelas = str(row[label_col]).replace(" ", "_")  # nama folder kelas
    geom = [mapping(row["geometry"])]

    # create folder for each class
    output_dir = os.path.join(output_root, kelas)
    os.makedirs(output_dir, exist_ok=True)

    try:
        # Clip raster according to polygon
        out_image, out_transform = mask(raster, geom, crop=True)
        out_meta = raster.meta.copy()

        # Update metadata
        out_meta.update({
            "driver": "GTiff",
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform
        })

        # save the clip result
        out_name = f"poly_{idx:04d}.tif"
        out_path = os.path.join(output_dir, out_name)
        with rasterio.open(out_path, "w", **out_meta) as dest:
            dest.write(out_image)

    except Exception as e:
        print(f"⚠️ Polygon {idx} is skipped (error: {e})")

raster.close()
print("All polygons are solved in clips and saved per class!")

100%|██████████| 185/185 [00:03<00:00, 56.36it/s]


All polygons are solved in clips and saved per class!
